In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import CamembertModel, AutoTokenizer, CLIPVisionModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

BATCH_SIZE = 16
ACCUM_STEPS = 2
MAX_TEXT_LEN = 128
NUM_WORKERS = 0
EARLY_STOPPING_PATIENCE = 5
RESUME = True

CAMEMBERT_ID = "camembert-base"
CLIP_ID = "openai/clip-vit-base-patch32"

STAGE_CONFIGS = [
    {
        "name": "stage1_head_only",
        "epochs": 3,
        "backbone_lr": 0.0,
        "head_lr": 5e-4
    },
    {
        "name": "stage2_partial_unfreeze",
        "epochs": 4,
        "backbone_lr": 2e-6,
        "head_lr": 1e-4
    },
    {
        "name": "stage3_full_unfreeze",
        "epochs": 8,
        "backbone_lr": 1e-6,
        "head_lr": 5e-5
    }
]

BASE_DIR = Path.cwd().parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"

RUN_NAME = "mm_camembert_clip_gated_fusion_staged_unfreeze"
OUTPUT_DIR = BASE_DIR / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LAST_CHECKPOINT_PATH = OUTPUT_DIR / "last_checkpoint.pt"
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "best_checkpoint.pt"
BEST_MODEL_PATH = OUTPUT_DIR / "best_model_state_dict.pt"
BEST_VAL_LOGITS_PATH = OUTPUT_DIR / "best_val_logits.npy"
HISTORY_CSV_PATH = OUTPUT_DIR / "training_history.csv"
PLOT_PATH = OUTPUT_DIR / "metrics_curves.png"
BEST_REPORT_PATH = OUTPUT_DIR / "best_classification_report.txt"
BEST_METADATA_PATH = OUTPUT_DIR / "best_metadata.json"

train_df = pd.read_csv(SPLIT_DIR / "train_split.csv")
val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json", "r", encoding="utf-8") as f:
    raw_label2id = json.load(f)
    label2id = {int(k): int(v) for k, v in raw_label2id.items()}

id2label = {v: k for k, v in label2id.items()}
NUM_CLASSES = len(label2id)

tokenizer = AutoTokenizer.from_pretrained(CAMEMBERT_ID)
processor = CLIPProcessor.from_pretrained(CLIP_ID)


class RakutenDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = IMAGE_DIR / f"image_{row['imageid']}_product_{row['productid']}.jpg"
        image = Image.open(img_path).convert("RGB")

        designation = "" if pd.isna(row.get("designation", "")) else str(row.get("designation", ""))
        description = "" if pd.isna(row.get("description", "")) else str(row.get("description", ""))
        text = f"{designation} {description}".strip()

        txt = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=MAX_TEXT_LEN,
            return_tensors="pt"
        )

        img = processor(images=image, return_tensors="pt")

        return {
            "input_ids": txt["input_ids"].squeeze(0),
            "attention_mask": txt["attention_mask"].squeeze(0),
            "pixel_values": img["pixel_values"].squeeze(0),
            "labels": torch.tensor(label2id[int(row["prdtypecode"])], dtype=torch.long)
        }


train_loader = DataLoader(
    RakutenDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    RakutenDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.text = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.vision = CLIPVisionModel.from_pretrained(CLIP_ID)

        self.gate = nn.Sequential(
            nn.Linear(1536, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 768),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(2304, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

        self.debug_done = False

    def forward(self, input_ids, attention_mask, pixel_values):
        text_outputs = self.text(input_ids=input_ids, attention_mask=attention_mask)
        vision_outputs = self.vision(pixel_values=pixel_values)

        t = text_outputs.last_hidden_state[:, 0]
        v = vision_outputs.pooler_output

        t = t / t.norm(dim=-1, keepdim=True).clamp(min=1e-12)
        v = v / v.norm(dim=-1, keepdim=True).clamp(min=1e-12)

        fusion = torch.cat([t, v], dim=1)
        g = self.gate(fusion)

        fused = g * v + (1.0 - g) * t
        final = torch.cat([fused, t, v], dim=1)

        if not self.debug_done:
            print("text embedding shape:", t.shape)
            print("vision embedding shape:", v.shape)
            print("fusion shape:", fusion.shape)
            print("gate shape:", g.shape)
            print("final shape:", final.shape)
            self.debug_done = True

        return self.classifier(final)


model = FusionModel(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()


def freeze_module(module):
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_module(module):
    for p in module.parameters():
        p.requires_grad = True


def freeze_all_backbones(model):
    freeze_module(model.text)
    freeze_module(model.vision)


def unfreeze_last_camembert_layers(model, layer_indices=(10, 11)):
    for name, p in model.text.named_parameters():
        if any(f"encoder.layer.{idx}." in name for idx in layer_indices):
            p.requires_grad = True


def unfreeze_last_clip_layers(model, layer_indices=(10, 11)):
    for name, p in model.vision.named_parameters():
        if any(f"vision_model.encoder.layers.{idx}." in name for idx in layer_indices):
            p.requires_grad = True
        elif "vision_model.post_layernorm" in name:
            p.requires_grad = True


def apply_stage_freezing(model, stage_name):
    unfreeze_module(model.gate)
    unfreeze_module(model.classifier)

    if stage_name == "stage1_head_only":
        freeze_all_backbones(model)
    elif stage_name == "stage2_partial_unfreeze":
        freeze_all_backbones(model)
        unfreeze_last_camembert_layers(model, layer_indices=(10, 11))
        unfreeze_last_clip_layers(model, layer_indices=(10, 11))
    elif stage_name == "stage3_full_unfreeze":
        unfreeze_module(model)
    else:
        raise ValueError(f"Unknown stage name: {stage_name}")


def build_optimizer(model, backbone_lr, head_lr):
    backbone_params = []
    head_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("gate") or name.startswith("classifier"):
            head_params.append(p)
        else:
            backbone_params.append(p)

    param_groups = []
    if backbone_params and backbone_lr > 0:
        param_groups.append({"params": backbone_params, "lr": backbone_lr})
    if head_params:
        param_groups.append({"params": head_params, "lr": head_lr})

    return torch.optim.AdamW(param_groups, weight_decay=1e-4)


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def run_epoch(model, loader, optimizer=None, train=True):
    if train:
        model.train()
        optimizer.zero_grad(set_to_none=True)
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_true = []
    all_logits = []

    for step, batch in enumerate(tqdm(loader, leave=False), start=1):
        if step == 1:
            print("first batch arrived")
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.set_grad_enabled(train):
            logits, gate_weights = model(input_ids, attention_mask, pixel_values)

            if step == 1:
              print("forward done")

            loss = criterion(logits, labels)

            if train:
                (loss / ACCUM_STEPS).backward()

                if step % ACCUM_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * labels.size(0)

        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.detach().cpu().numpy())

        if not train:
            all_logits.append(logits.detach().cpu().numpy())

    if train and (len(loader) % ACCUM_STEPS != 0):
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_true, all_preds)
    epoch_f1 = f1_score(all_true, all_preds, average="macro")
    logits_out = np.concatenate(all_logits, axis=0) if not train else None

    return epoch_loss, epoch_acc, epoch_f1, logits_out, all_true, all_preds


def save_checkpoint(path, model, optimizer, history, global_epoch, stage_idx, local_epoch, best_f1, best_epoch_global, best_stage_name, epochs_without_improvement):
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "history": history,
        "global_epoch": global_epoch,
        "stage_idx": stage_idx,
        "local_epoch": local_epoch,
        "best_f1": best_f1,
        "best_epoch_global": best_epoch_global,
        "best_stage_name": best_stage_name,
        "epochs_without_improvement": epochs_without_improvement,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "accum_steps": ACCUM_STEPS,
        "max_text_len": MAX_TEXT_LEN,
        "camembert_id": CAMEMBERT_ID,
        "clip_id": CLIP_ID,
        "stage_configs": STAGE_CONFIGS
    }
    torch.save(payload, path)


def load_checkpoint(path):
    return torch.load(path, map_location=DEVICE)


def plot_history(history, output_path):
    if len(history) == 0:
        return

    df = pd.DataFrame(history)

    plt.figure(figsize=(10, 6))
    plt.plot(df["global_epoch"], df["train_acc"], label="Train Accuracy")
    plt.plot(df["global_epoch"], df["val_acc"], label="Validation Accuracy")
    plt.plot(df["global_epoch"], df["train_macro_f1"], label="Train Macro F1")
    plt.plot(df["global_epoch"], df["val_macro_f1"], label="Validation Macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Training Metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


history = []
best_f1 = -1.0
best_epoch_global = -1
best_stage_name = None
epochs_without_improvement = 0
global_epoch = 0
start_stage_idx = 0
resume_local_epoch = 0

if RESUME and LAST_CHECKPOINT_PATH.exists():
    print(f"Resuming from checkpoint: {LAST_CHECKPOINT_PATH}")
    ckpt = load_checkpoint(LAST_CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model_state_dict"])
    history = ckpt["history"]
    global_epoch = ckpt["global_epoch"]
    start_stage_idx = ckpt["stage_idx"]
    resume_local_epoch = ckpt["local_epoch"]
    best_f1 = ckpt["best_f1"]
    best_epoch_global = ckpt["best_epoch_global"]
    best_stage_name = ckpt["best_stage_name"]
    epochs_without_improvement = ckpt["epochs_without_improvement"]
else:
    print("Starting training from scratch")

stop_training = False

for stage_idx in range(start_stage_idx, len(STAGE_CONFIGS)):
    stage_cfg = STAGE_CONFIGS[stage_idx]
    stage_name = stage_cfg["name"]
    stage_epochs = stage_cfg["epochs"]
    backbone_lr = stage_cfg["backbone_lr"]
    head_lr = stage_cfg["head_lr"]

    print("\n" + "=" * 80)
    print(f"Starting {stage_name}")
    print("=" * 80)

    apply_stage_freezing(model, stage_name)
    optimizer = build_optimizer(model, backbone_lr=backbone_lr, head_lr=head_lr)

    if RESUME and LAST_CHECKPOINT_PATH.exists() and stage_idx == start_stage_idx:
        ckpt = load_checkpoint(LAST_CHECKPOINT_PATH)
        if ckpt["optimizer_state_dict"] is not None:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    trainable, total = count_trainable_params(model)
    print(f"Trainable params: {trainable:,} / {total:,}")
    print(f"Backbone LR: {backbone_lr}")
    print(f"Head LR: {head_lr}")

    stage_start_epoch = resume_local_epoch + 1 if stage_idx == start_stage_idx else 1

    for local_epoch in range(stage_start_epoch, stage_epochs + 1):
        global_epoch += 1

        train_loss, train_acc, train_f1, _, _, _ = run_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            train=True
        )

        val_loss, val_acc, val_f1, val_logits, y_true, y_pred = run_epoch(
            model=model,
            loader=val_loader,
            optimizer=None,
            train=False
        )

        print(
            f"Epoch {global_epoch:02d} | "
            f"Stage={stage_name} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"val_f1={val_f1:.4f}"
        )

        history.append({
            "global_epoch": global_epoch,
            "stage_idx": stage_idx,
            "stage": stage_name,
            "stage_epoch": local_epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_macro_f1": val_f1,
            "backbone_lr": backbone_lr,
            "head_lr": head_lr
        })

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        plot_history(history, PLOT_PATH)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch_global = global_epoch
            best_stage_name = stage_name
            epochs_without_improvement = 0

            torch.save(model.state_dict(), BEST_MODEL_PATH)
            np.save(BEST_VAL_LOGITS_PATH, val_logits)

            report = classification_report(y_true, y_pred, digits=4)
            with open(BEST_REPORT_PATH, "w", encoding="utf-8") as f:
                f.write(report)

            best_metadata = {
                "best_val_macro_f1": float(best_f1),
                "best_epoch_global": int(best_epoch_global),
                "best_stage_name": best_stage_name,
                "num_classes": int(NUM_CLASSES),
                "batch_size": int(BATCH_SIZE),
                "accum_steps": int(ACCUM_STEPS),
                "max_text_len": int(MAX_TEXT_LEN),
                "camembert_id": CAMEMBERT_ID,
                "clip_id": CLIP_ID,
                "device": str(DEVICE)
            }
            with open(BEST_METADATA_PATH, "w", encoding="utf-8") as f:
                json.dump(best_metadata, f, indent=2, ensure_ascii=False)

            save_checkpoint(
                path=BEST_CHECKPOINT_PATH,
                model=model,
                optimizer=optimizer,
                history=history,
                global_epoch=global_epoch,
                stage_idx=stage_idx,
                local_epoch=local_epoch,
                best_f1=best_f1,
                best_epoch_global=best_epoch_global,
                best_stage_name=best_stage_name,
                epochs_without_improvement=epochs_without_improvement
            )

            print(f"  -> New best model saved. val_f1={val_f1:.4f}")

        else:
            epochs_without_improvement += 1
            print(f"  -> No improvement for {epochs_without_improvement} epoch(s)")

        save_checkpoint(
            path=LAST_CHECKPOINT_PATH,
            model=model,
            optimizer=optimizer,
            history=history,
            global_epoch=global_epoch,
            stage_idx=stage_idx,
            local_epoch=local_epoch,
            best_f1=best_f1,
            best_epoch_global=best_epoch_global,
            best_stage_name=best_stage_name,
            epochs_without_improvement=epochs_without_improvement
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            stop_training = True
            break

    resume_local_epoch = 0

    if stop_training:
        break

print("\nTraining finished.")
print(f"Best global epoch: {best_epoch_global}")
print(f"Best stage: {best_stage_name}")
print(f"Best val macro F1: {best_f1:.4f}")
print(f"History CSV: {HISTORY_CSV_PATH}")
print(f"Plot: {PLOT_PATH}")
print(f"Best model weights: {BEST_MODEL_PATH}")
print(f"Last checkpoint: {LAST_CHECKPOINT_PATH}")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epochs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

train_loss = [0.8687, 0.6643, 0.6037, 0.4801, 0.4143, 0.3564, 0.3067, 0.2629, 0.1903, 0.1368, 0.0931]
train_acc  = [0.7352, 0.7893, 0.8054, 0.8429, 0.8632, 0.8822, 0.8988, 0.9126, 0.9394, 0.9593, 0.9730]
train_f1   = [0.7064, 0.7744, 0.7936, 0.8346, 0.8575, 0.8778, 0.8959, 0.9084, 0.9394, 0.9594, 0.9733]

val_loss = [0.5891, 0.5304, 0.5241, 0.4552, 0.4368, 0.4171, 0.4072, 0.3977, 0.3992, 0.4193, 0.4360]
val_acc  = [0.8092, 0.8268, 0.8263, 0.8504, 0.8577, 0.8654, 0.8697, 0.8752, 0.8766, 0.8751, 0.8743]
val_f1   = [0.7971, 0.8129, 0.8145, 0.8408, 0.8465, 0.8527, 0.8571, 0.8634, 0.8642, 0.8644, 0.8626]

best_epoch = 10
best_val_f1 = 0.8644

plt.figure(figsize=(8, 5), facecolor="white")
ax = plt.gca()
ax.set_facecolor("white")

plt.plot(epochs, train_f1, marker="o", label="Train Macro F1")
plt.plot(epochs, val_f1, marker="o", label="Validation Macro F1")

plt.scatter(best_epoch, best_val_f1, s=100)
plt.text(best_epoch, best_val_f1 + 0.002, f"Best: {best_val_f1:.4f}", ha="center")

plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Training and Validation Macro F1")
plt.legend()
plt.grid(False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()